# Clean Dataset

This notebook applies the fixes identified in `dataset_eda.ipynb` and saves the cleaned dataset used by every model notebook.

It works on whole rows and on one source fingerprint (the Reuters dateline). Text processing such as lowercasing, punctuation removal and lemmatization is not done here; it stays in `preprocessing.py`.

## 1. Setup

In [1]:
import pandas as pd

## 2. Load the Combined Dataset

In [2]:
df = pd.read_csv("Dataset/fake_news_combined.csv.gz")
n_start = len(df)

print("Shape:", df.shape)
print(df["label"].value_counts())

Shape: (44898, 6)
label
fake    23481
real    21417
Name: count, dtype: int64


## 3. Strip Extra Spaces

Titles, texts and dates have leading or trailing spaces that differ between the two files. Stripping them first means that the checks for empty texts and duplicates below are not thrown off by spacing.

In [3]:
for col in ["title", "text", "date"]:
    df[col] = df[col].str.strip()

## 4. Remove the Reuters Dateline

99.2% of real articles start with a dateline such as "WASHINGTON (Reuters) - ", sometimes after a correction note or a byline, and no fake article does. It identifies the publisher, not the content, so it is removed. The rest of the article is kept.

The output below confirms it: 21,246 datelines are removed, 99.2% of the 21,417 real articles.

In [4]:
DATELINE = r"^\s*.{0,300}?\(Reuters\)\s*[-–—]\s*"

has_dateline = df["text"].str.contains(DATELINE, regex=True)
example = df.loc[has_dateline, "text"].iloc[0]

df["text"] = df["text"].str.replace(DATELINE, "", regex=True).str.strip()

print(f"Datelines removed: {has_dateline.sum()}")
print("\nBEFORE:", example[:150])
print("AFTER: ", pd.Series([example]).str.replace(DATELINE, "", regex=True).str.strip().iloc[0][:150])

Datelines removed: 21246

BEFORE: SAN FRANCISCO (Reuters) - California Attorney General Xavier Becerra said on Friday he was “prepared to take whatever action it takes” to defend the O
AFTER:  California Attorney General Xavier Becerra said on Friday he was “prepared to take whatever action it takes” to defend the Obamacare mandate that heal


## 5. Drop Rows with an Invalid Date

10 fake rows have a URL or a headline in the date column. These were scraped incorrectly, so they are dropped. All other dates are converted to a proper date type.

In [5]:
df["date"] = pd.to_datetime(df["date"], format="mixed", errors="coerce")

n_before = len(df)
df = df[df["date"].notna()]
print(f"Dropped {n_before - len(df)} rows with an invalid date")

Dropped 10 rows with an invalid date


## 6. Drop Empty Texts

Articles with no text give a model nothing to read except the title. This check runs after the dateline is removed, so an article that contained only a dateline would also be caught.

In [6]:
n_before = len(df)
df = df[df["text"] != ""]
print(f"Dropped {n_before - len(df)} rows with empty text")

Dropped 631 rows with empty text


## 7. Drop Duplicate Articles

When the same text appears more than once, only the first copy is kept. This runs after the dateline is removed, so two copies of a story that differ only in their dateline are also caught. It has to happen before the train/test split, so that no article can appear in both.

In [7]:
n_before = len(df)
df = df.drop_duplicates(subset="text", keep="first").reset_index(drop=True)
print(f"Dropped {n_before - len(df)} duplicate articles")

Dropped 5624 duplicate articles


## 8. Check the Result

In [8]:
print(f"Rows: {n_start} -> {len(df)} ({n_start - len(df)} removed)\n")

print("Missing values:", df.isnull().sum().sum())
print("Empty texts:", (df["text"] == "").sum())
print("Duplicate texts:", df["text"].duplicated().sum())
print("Datelines left:", df["text"].str.contains(DATELINE, regex=True).sum())

print("\nClass balance:")
print(df["label"].value_counts())
print(df["label"].value_counts(normalize=True).mul(100).round(1))

Rows: 44898 -> 38633 (6265 removed)

Missing values: 0
Empty texts: 0
Duplicate texts: 0
Datelines left: 1

Class balance:
label
real    21190
fake    17443
Name: count, dtype: int64
label
real    54.8
fake    45.2
Name: proportion, dtype: float64


6,265 rows (14%) were removed, leaving 38,633 articles. Most of the loss is duplicates (5,624), almost all of them fake, so the share of fake articles falls from 52.3% to 45.2%. The classes are still close to balanced, and the stratified split keeps the same ratio in training and testing.

One article still matches the dateline pattern. It is a "Highlights of the day" roundup that opens with two datelines, so only the first was removed. A single article has no meaningful effect, so it is left as it is.

Always predicting "real" now gets 54.8% accuracy, which is the baseline every model has to beat.

## 9. Save

`subject` and `date` are kept for reference but must not be used as model features, since each gives away the label (see `dataset_eda.ipynb`). The file is gzip-compressed to stay under GitHub's 100 MB limit.

In [9]:
df.to_csv("Dataset/fake_news_dataset_cleaned.csv.gz", index=False, compression="gzip")
print("Saved cleaned dataset to Dataset/fake_news_dataset_cleaned.csv.gz")

Saved cleaned dataset to Dataset/fake_news_dataset_cleaned.csv.gz
